In [2]:
# ============================================================================
# Cohort construction + missingness diagnostics
# ============================================================================
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

# ----------------------------------------------------------------------------
# 1. Load
# ----------------------------------------------------------------------------
DATA_PATH = Path(
    "../../Dataset/mimic-iv/from_pipeline/ltm_grided_clipped_A_D_Z_Y_SelfPipeline.parquet"
)
mimicdata = pd.read_parquet(DATA_PATH).copy()

mimicdata = mimicdata.rename(
    columns={
        "stay_id": "admission_id",
        "t": "t0",
        "raw_age": "age",
        "raw_height": "height",
        "raw_weight": "weight",
    }
).copy()

categorical_cols = [
    "vent_mode__last__last_12h",
    "fio2__last__last_12h",
    "glasgow_coma_scale_total__last__last_12h",
]
for c in categorical_cols:
    if c in mimicdata.columns:
        mimicdata[c] = mimicdata[c].astype("category")

print("Raw shape:        ", mimicdata.shape)
print("Raw admissions:   ", mimicdata["admission_id"].nunique())
print("\nAll columns:")
print(list(mimicdata.columns))

# ----------------------------------------------------------------------------
# 2. Config
# ----------------------------------------------------------------------------
picked_L = [
    "vent_mode__last__last_12h",
    "vent_mode__hours_since_last__last_12h",
    "bicarbonate_arterial__last__last_12h",
    "activated_partial_thromboplastin_time__last__last_12h",
    "temperature__mean__last_12h",
    "hemoglobin__last__last_12h",
    "heart_rate__mean__last_12h",
    "arterial_blood_pressure_mean__mean__last_12h",
    "creatinine__last__last_12h",
    "ureum__last__last_12h",
    "fluid_out_urine__mean__last_12h",
    "lactate__last__last_12h",
    "glasgow_coma_scale_total__last__last_12h",
    "pco2_arterial__mean__last_12h",
    "respiratory_rate_measured__mean__last_12h",
    "o2_saturation__mean__last_12h",
    "fio2__last__last_12h",
    "po2_arterial__mean__last_12h",
]
baseline   = ["age", "sex", "origin"]
id_        = "admission_id"
time_name  = "t0"

# EDIT THESE after inspecting the column list above:
COL_I = "D"   # in-ICU death  (time-varying or admission-level indicator)
COL_Z = "Z"   # post-discharge death
COL_A = "A"   # discharge indicator (time-varying)
COL_Y = "Y"   # if you have a separate Y; otherwise leave as is or set None
COL_FOLLOWUP_DAYS = None   # set to your follow-up column if available

# Sanity: surface which of these are actually present
for c in [COL_I, COL_Z, COL_A, COL_Y]:
    present = c in mimicdata.columns
    print(f"  {c}: {'found' if present else 'MISSING'}")

# Severity proxies you may have at admission level. Add as available.
SEVERITY_PROXIES_CANDIDATES = ["height", "weight"]  # extend if you have los, icu_unit, etc.
SEVERITY_PROXIES = [c for c in SEVERITY_PROXIES_CANDIDATES if c in mimicdata.columns]

# t0=0 values of L variables — these are partially observed even in the
# excluded cohort and are the most informative observable severity proxies.
PICKED_L_AT_T0 = [
    "heart_rate__mean__last_12h",
    "arterial_blood_pressure_mean__mean__last_12h",
    "respiratory_rate_measured__mean__last_12h",
    "o2_saturation__mean__last_12h",
    "temperature__mean__last_12h",
    "glasgow_coma_scale_total__last__last_12h",
]

# ----------------------------------------------------------------------------
# 3. Identify excluded admissions WITHOUT mutating the full frame
#    (replicates your original logic: ffill+bfill within admission, then
#     drop admissions with any residual NA in picked_L or baseline)
# ----------------------------------------------------------------------------
mimicdata = mimicdata.sort_values([id_, time_name]).copy()

# Compute filled L on a copy, only to identify bad admissions
filled_L = mimicdata.groupby(id_)[picked_L].ffill().bfill()
required_check = pd.concat(
    [filled_L, mimicdata[baseline]], axis=1
)
bad_mask = required_check.isna().any(axis=1)
bad_admissions = mimicdata.loc[bad_mask, id_].unique()

n_total_adm = mimicdata[id_].nunique()
n_bad       = len(bad_admissions)
n_kept      = n_total_adm - n_bad
print(f"\nTotal admissions:        {n_total_adm}")
print(f"Excluded (R=0):          {n_bad} ({n_bad/n_total_adm:.4f})")
print(f"Included (R=1):          {n_kept}")

# ----------------------------------------------------------------------------
# 4. Build admission-level frame for diagnostics (uses PRE-exclusion data)
# ----------------------------------------------------------------------------
def collapse_to_admission(df_full, picked_L_at_t0):
    """One row per admission with baseline, t0=0 values, severity proxies,
    and admission-level outcome summaries. Uses pre-exclusion data."""
    df = df_full.sort_values([id_, time_name])

    # First-row (t0 = min) values for baseline + L-at-t0 + severity proxies
    first_cols = list(set(baseline + SEVERITY_PROXIES + picked_L_at_t0))
    first_cols = [c for c in first_cols if c in df.columns]
    first_row  = df.groupby(id_, as_index=False)[first_cols].first()

    # Outcome summaries — robust to time-varying or admission-level encoding.
    # I, Z, A treated as: did the event ever occur during the admission?
    out_summaries = {}
    for col, name in [(COL_I, "ever_in_icu_death"),
                      (COL_Z, "ever_post_disch_death"),
                      (COL_A, "ever_discharged")]:
        if col in df.columns:
            agg = df.groupby(id_)[col].max().astype(float).reset_index()
            agg = agg.rename(columns={col: name})
            out_summaries[name] = agg

    # Optional follow-up
    if COL_FOLLOWUP_DAYS and COL_FOLLOWUP_DAYS in df.columns:
        fu = df.groupby(id_)[COL_FOLLOWUP_DAYS].max().reset_index()
        out_summaries["followup_days"] = fu

    # Length of admission in t0 units (a useful severity proxy)
    los = df.groupby(id_)[time_name].agg(lambda s: s.max() - s.min() + 1).reset_index()
    los = los.rename(columns={time_name: "n_t_obs"})
    out_summaries["n_t_obs"] = los

    adm = first_row
    for _, df_o in out_summaries.items():
        adm = adm.merge(df_o, on=id_, how="left")

    adm["R"] = (~adm[id_].isin(set(bad_admissions))).astype(int)
    return adm

adm = collapse_to_admission(mimicdata, PICKED_L_AT_T0)
print(f"\nAdmission-level frame: {adm.shape}")
print(f"  R=1 (included): {(adm['R']==1).sum()}")
print(f"  R=0 (excluded): {(adm['R']==0).sum()}")

# ----------------------------------------------------------------------------
# 5. DIAGNOSTIC 1 — Standardised mean differences
# ----------------------------------------------------------------------------
def smd_continuous(x_inc, x_exc):
    x_inc = pd.to_numeric(pd.Series(x_inc), errors="coerce").dropna()
    x_exc = pd.to_numeric(pd.Series(x_exc), errors="coerce").dropna()
    if len(x_inc) < 2 or len(x_exc) < 2:
        return np.nan, len(x_inc), len(x_exc)
    pooled = np.sqrt((x_inc.var(ddof=1) + x_exc.var(ddof=1)) / 2.0)
    if pooled == 0:
        return 0.0, len(x_inc), len(x_exc)
    return (x_inc.mean() - x_exc.mean()) / pooled, len(x_inc), len(x_exc)

def smd_binary(p1, p0):
    pooled = np.sqrt((p1*(1-p1) + p0*(1-p0)) / 2.0)
    if pooled == 0 or np.isnan(pooled):
        return 0.0
    return (p1 - p0) / pooled

def smd_table(adm, variables):
    rows = []
    inc = adm[adm["R"] == 1]
    exc = adm[adm["R"] == 0]
    for v in variables:
        if v not in adm.columns:
            continue
        s = adm[v]
        # Categorical with object/category dtype
        if s.dtype == "object" or pd.api.types.is_categorical_dtype(s):
            levels = s.dropna().unique()
            for lvl in levels:
                p1 = (inc[v] == lvl).mean()
                p0 = (exc[v] == lvl).mean()
                rows.append({
                    "variable": f"{v}={lvl}",
                    "type": "categorical",
                    "mean_included": p1,
                    "mean_excluded": p0,
                    "SMD": smd_binary(p1, p0),
                    "n_included": inc[v].notna().sum(),
                    "n_excluded": exc[v].notna().sum(),
                })
        # Binary numeric
        elif set(pd.to_numeric(s, errors="coerce").dropna().unique()).issubset({0, 1, 0.0, 1.0}):
            p1 = pd.to_numeric(inc[v], errors="coerce").mean()
            p0 = pd.to_numeric(exc[v], errors="coerce").mean()
            rows.append({
                "variable": v,
                "type": "binary",
                "mean_included": p1,
                "mean_excluded": p0,
                "SMD": smd_binary(p1, p0),
                "n_included": inc[v].notna().sum(),
                "n_excluded": exc[v].notna().sum(),
            })
        else:
            d, n1, n0 = smd_continuous(inc[v], exc[v])
            rows.append({
                "variable": v,
                "type": "continuous",
                "mean_included": pd.to_numeric(inc[v], errors="coerce").mean(),
                "mean_excluded": pd.to_numeric(exc[v], errors="coerce").mean(),
                "SMD": d,
                "n_included": n1,
                "n_excluded": n0,
            })
    out = pd.DataFrame(rows)
    out["abs_SMD"] = out["SMD"].abs()
    return out.sort_values("abs_SMD", ascending=False).reset_index(drop=True)

variables_to_compare = (
    baseline
    + SEVERITY_PROXIES
    + ["n_t_obs"]
    + [c for c in PICKED_L_AT_T0 if c in adm.columns]
)
smd_tab = smd_table(adm, variables_to_compare)

print("\n=== DIAGNOSTIC 1: Standardised mean differences (included vs excluded) ===")
print("Convention: |SMD| < 0.10 reassuring, 0.10-0.25 discuss, > 0.25 problem")
with pd.option_context("display.max_rows", None, "display.width", 200,
                        "display.float_format", lambda x: f"{x:.4f}"):
    print(smd_tab.to_string(index=False))

# ----------------------------------------------------------------------------
# 6. DIAGNOSTIC 2 — Outcome rates with Wilson 95% CIs and bootstrap RD CIs
# ----------------------------------------------------------------------------
def wilson_ci(k, n, alpha=0.05):
    if n == 0:
        return (np.nan, np.nan)
    z = stats.norm.ppf(1 - alpha/2)
    phat = k / n
    denom = 1 + z**2 / n
    centre = (phat + z**2/(2*n)) / denom
    half = (z * np.sqrt(phat*(1-phat)/n + z**2/(4*n**2))) / denom
    return (centre - half, centre + half)

def risk_diff_boot_ci(k1, n1, k0, n0, alpha=0.05, B=5000, seed=0):
    if n1 == 0 or n0 == 0:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    p1_b = rng.binomial(n1, k1/n1, size=B) / n1
    p0_b = rng.binomial(n0, k0/n0, size=B) / n0
    diff = p1_b - p0_b
    return tuple(np.quantile(diff, [alpha/2, 1 - alpha/2]))

def risk_ratio_ci(k1, n1, k0, n0, alpha=0.05):
    if n1 == 0 or n0 == 0:
        return (np.nan, np.nan, np.nan)
    if k1 == 0 or k0 == 0:
        a, b, c, d = k1+0.5, n1-k1+0.5, k0+0.5, n0-k0+0.5
        rr = (a/(a+b)) / (c/(c+d))
        se = np.sqrt(1/a - 1/(a+b) + 1/c - 1/(c+d))
    else:
        rr = (k1/n1) / (k0/n0)
        se = np.sqrt((1-k1/n1)/k1 + (1-k0/n0)/k0)
    z = stats.norm.ppf(1 - alpha/2)
    return (rr, np.exp(np.log(rr) - z*se), np.exp(np.log(rr) + z*se))

def outcome_compare(adm, outcome_col, label):
    if outcome_col not in adm.columns:
        return None
    sub = adm[[outcome_col, "R"]].dropna()
    inc = sub[sub["R"] == 1][outcome_col].astype(int)
    exc = sub[sub["R"] == 0][outcome_col].astype(int)
    n1, k1 = len(inc), int(inc.sum())
    n0, k0 = len(exc), int(exc.sum())
    p1 = k1/n1 if n1 else np.nan
    p0 = k0/n0 if n0 else np.nan
    ci1 = wilson_ci(k1, n1)
    ci0 = wilson_ci(k0, n0)
    rd_lo, rd_hi = risk_diff_boot_ci(k1, n1, k0, n0)
    rr, rr_lo, rr_hi = risk_ratio_ci(k1, n1, k0, n0)
    return {
        "outcome": label,
        "n_included": n1, "events_included": k1, "rate_included": p1,
        "wilson_inc": f"[{ci1[0]:.4f}, {ci1[1]:.4f}]",
        "n_excluded": n0, "events_excluded": k0, "rate_excluded": p0,
        "wilson_exc": f"[{ci0[0]:.4f}, {ci0[1]:.4f}]",
        "risk_diff_inc_minus_exc": (p1 - p0) if (n0 and n1) else np.nan,
        "rd_95CI": f"[{rd_lo:.4f}, {rd_hi:.4f}]",
        "risk_ratio": rr,
        "rr_95CI": f"[{rr_lo:.3f}, {rr_hi:.3f}]",
    }

print("\n=== DIAGNOSTIC 2: Admission-level outcome rates ===")
rows = []
for col, lbl in [("ever_in_icu_death",     "I  (in-ICU death)"),
                 ("ever_post_disch_death", "Z  (post-discharge death)"),
                 ("ever_discharged",       "A  (ever discharged)")]:
    r = outcome_compare(adm, col, lbl)
    if r is not None:
        rows.append(r)
outcome_tab = pd.DataFrame(rows)
with pd.option_context("display.max_columns", None, "display.width", 220,
                        "display.float_format", lambda x: f"{x:.4f}"):
    print(outcome_tab.to_string(index=False))

# ----------------------------------------------------------------------------
# 7. Save
# ----------------------------------------------------------------------------
smd_tab.to_csv("diagnostic1_smd.csv", index=False)
outcome_tab.to_csv("diagnostic2_outcomes.csv", index=False)
print("\nSaved: diagnostic1_smd.csv, diagnostic2_outcomes.csv")

# ----------------------------------------------------------------------------
# 8. Now apply the actual exclusion (your original step) for downstream code
# ----------------------------------------------------------------------------
mimicdata[picked_L] = mimicdata.groupby(id_)[picked_L].ffill().bfill()
mimicdata = mimicdata[~mimicdata[id_].isin(bad_admissions)].copy()
print(f"\nAfter exclusion — shape: {mimicdata.shape}, "
      f"admissions: {mimicdata[id_].nunique()}")

Raw shape:         (701079, 40)
Raw admissions:    85037

All columns:
['admission_id', 'grid_end', 'vent_mode__hours_since_last__last_12h', 'temperature__mean__last_12h', 'heart_rate__mean__last_12h', 'arterial_blood_pressure_mean__mean__last_12h', 'fluid_out_urine__mean__last_12h', 'pco2_arterial__mean__last_12h', 'respiratory_rate_measured__mean__last_12h', 'o2_saturation__mean__last_12h', 'po2_arterial__mean__last_12h', 'bicarbonate_arterial__last__last_12h', 'activated_partial_thromboplastin_time__last__last_12h', 'hemoglobin__last__last_12h', 'creatinine__last__last_12h', 'ureum__last__last_12h', 'lactate__last__last_12h', 'glasgow_coma_scale_total__last__last_12h', 'fio2__last__last_12h', 'vent_mode__last__last_12h', 'subject_id', 'age', 'sex', 'height', 'weight', 'unit_type', 'origin', 'los', 'intime', 'outtime', 'death_time_from_intime', 'icu_mortality_flag', 'death_abs_time', 'icu_mortality', 't0', 'mortality_after_discharge_30d', 'A', 'D', 'Z', 'Y']
  D: found
  Z: found
  A

/tmp/ipykernel_644050/3499712529.py:191: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if s.dtype == "object" or pd.api.types.is_categorical_dtype(s):



After exclusion — shape: (672354, 40), admissions: 81890


In [3]:
baseline_no_origin = ["age", "sex"]
required_no_origin = picked_L + baseline_no_origin

filled_L_check = mimicdata.groupby(id_)[picked_L].ffill().bfill()
required_check_no_origin = pd.concat(
    [filled_L_check, mimicdata[baseline_no_origin]], axis=1
)
bad_mask_no_origin = required_check_no_origin.isna().any(axis=1)
bad_admissions_no_origin = mimicdata.loc[bad_mask_no_origin, id_].unique()

print(f"With origin requirement:    {len(bad_admissions)} excluded ({len(bad_admissions)/85037:.4f})")
print(f"Without origin requirement: {len(bad_admissions_no_origin)} excluded "
      f"({len(bad_admissions_no_origin)/85037:.4f})")

# How many of the original 3147 are recovered if we drop the origin requirement?
recovered = set(bad_admissions) - set(bad_admissions_no_origin)
print(f"Admissions recovered by treating origin=Unknown as a level: {len(recovered)}")

With origin requirement:    3147 excluded (0.0370)
Without origin requirement: 0 excluded (0.0000)
Admissions recovered by treating origin=Unknown as a level: 3147
